**Eye-Tracking & Neurocognitive Intelligence | Eye-Tracking Foundations & Signal Analysis | Presented by: Reza Saadatyar (2026)**

**01 | GazeBase Source Audit and Gaze Quality**

![Conceptual workflow for stage 01](../../images/01_source_audit_and_gaze_quality.svg)

**Objective:** Inspect actual GazeBase sample records, source identity, timing, validity and exploratory signal quality.

**Dataset:** GazeBase v2.0 source CSV; exact selected files determine the observed sample.

**Notebook scope:** No event detector or cognitive label is fitted. This stage creates explicit, reusable sample-level current outputs.

**Execution status:** Source code prepared. No original-data results are claimed until this notebook is executed with the specified release.


**00. Colab Repository Setup**

In [1]:
# ======================== 00. Colab Repository Setup =========================

# -------------------------- 1. Repository Checkout ---------------------------
from pathlib import Path
import subprocess

repo_url = "https://github.com/RezaSaadatyar/Eye-Tracking-Neurocognitive-Intelligence.git"
repo_path = Path("/content") / repo_url.rsplit("/", 1)[-1].removesuffix(".git")
if not repo_path.exists():
    subprocess.run(["git", "clone", "-q", repo_url, str(repo_path)], check=True)

# ---------------------------- 2. Open Repository -----------------------------
%cd {repo_path}


/content/Eye-Tracking-Neurocognitive-Intelligence


**01. Imports and Research Environment**

In [2]:
# =================== 01. Imports and Research Environment ====================

# ------------------------ 1. Study Environment Setup -------------------------
from python_modules.core.colab_setup import setup_colab
root = setup_colab("eye_tracking_foundations")

# ---------------------------- 2. Standard Library ----------------------------
from pathlib import Path

# ----------------- 3. Scientific Computing and Data Analysis -----------------
import numpy as np
import pandas as pd

# ------------------- 4. Visualisation and Notebook Display -------------------
import matplotlib.pyplot as plt
from IPython.display import display
from core.plot_style import plot_style, save_figure
from core.output_download import download_outputs_from_colab

# ------------------------ 5. Study Research Utilities ------------------------
from gaze_source import discover_records, load_selected_records
from gaze_quality import summarise_quality

# --------------------- 6. Output and Presentation Setup ----------------------
output_dir = root / "eye_tracking_foundations" / "output"
output_dir.mkdir(parents=True, exist_ok=True)
plot_style(
    font_family="DejaVu Sans", title_size=12.5,
    title_weight="normal", label_size=11.5,
    label_weight="normal", tick_size=11,
    legend_size=10, figure_dpi=100, figure_size=(10, 4),
)
print("Current outputs:", output_dir)


Repository up to date.
Current outputs: /content/Eye-Tracking-Neurocognitive-Intelligence/eye_tracking_foundations/output


**02. Select Original GazeBase Recordings**

The official distribution is large. Download or extract a limited set of original CSVs under
~data/raw/gazebase/~ before execution. The source files must retain their original names.
The initial example uses Round 1, Session 1, and tasks FXS, HSS, RAN and TEX.
Only participants with all four available task files enter this bounded demonstration.
This is not a population-representative cohort or a predefined sample count.


In [3]:
# ================== 02. Select Original GazeBase Recordings ==================

source_dir = root / "data" / "raw" / "gazebase"
inventory = discover_records(
    source_dir, round_id=1, session_id=1,
    tasks=("FXS", "HSS", "RAN", "TEX"), max_participants=3,
)
inventory["source_dataset"] = "GazeBase"
inventory["source_archive"] = "GazeBase_v2_0.zip"
inventory["dataset_doi"] = "10.6084/m9.figshare.12912257"
inventory.drop(columns=["path"]).to_csv(output_dir / "01_source_manifest.csv", index=False)
display(inventory.drop(columns=["path"]))


FileNotFoundError: GazeBase CSV directory missing: /content/Eye-Tracking-Neurocognitive-Intelligence/data/raw/gazebase. Extract authorised GazeBase CSV records under data/raw/gazebase/ first.

**03. Read Original Sample Columns and Record Identities**

Preserve original timestamp ~n~ in milliseconds, gaze ~x/y~ in degrees of visual angle,
~val~ validity, optional uncalibrated pupil area and EyeLink parser labels when present.
The source reference and sample index are retained for downstream alignment.


In [ ]:
# ========== 03. Read Original Sample Columns and Record Identities ===========

samples = load_selected_records(inventory)
assert not samples.duplicated(["record_id", "sample_index"]).any()
samples.to_csv(output_dir / "01_gaze_samples.csv", index=False)
print(f"Records: {len(inventory)}, raw samples: {len(samples):,}")
display(samples.head(8))


**04. Audit Sampling and Missingness**

Report observed sample intervals, duplicate/reversed timestamps, invalid gaze,
large time gaps and finite pupil coverage by record. A nominal 1-kHz device does
not prove every recorded interval is exactly 1 ms. No records are excluded here.


In [ ]:
# ==================== 04. Audit Sampling and Missingness =====================

quality = summarise_quality(samples)
quality.to_csv(output_dir / "01_quality_by_record.csv", index=False)
display(quality)


**05. Visualise Recorded Gaze and Quality**

The displayed trace is an actual selected source record, not a generated example.
Do not interpret the resulting pupil or gaze pattern as direct evidence of attention.


In [ ]:
# ================== 05. Visualise Recorded Gaze and Quality ==================

record_id = str(inventory.iloc[0].record_id)
trace = samples[samples.record_id.eq(record_id)].head(10000)
fig, axes = plt.subplots(2, 1, figsize=(11, 5), sharex=True)
axes[0].plot(trace.time_s, trace.x, lw=0.8, label="x (dva)")
axes[0].plot(trace.time_s, trace.y, lw=0.8, label="y (dva)")
axes[0].set_ylabel("Gaze (dva)")
axes[0].legend(loc="upper right")
axes[1].step(trace.time_s, trace.sample_valid.astype(int), where="post")
axes[1].set_ylabel("Valid sample")
axes[1].set_xlabel("Time from record start (s)")
fig.suptitle(f"GazeBase source and validity: {record_id}")
fig.tight_layout()
save_figure(fig, output_dir / "01_gaze_quality.png")
plt.show()


**06. Scientific Summary and Output Download**

Notebook 02 consumes ~01_gaze_samples.csv~ directly; no repeat parsing of the
source archive is required. Save the ZIP, and if the Colab runtime changes,
upload that exact stage ZIP before running Notebook 02. The original CSV
files remain external and are never republished automatically.


In [ ]:
# ================ 06. Scientific Summary and Output Download =================

print("Saved source manifest, raw sample table, record quality and evidence plot.")
print("Observed participants:", quality.participant_id.nunique())
print("Valid-sample range:", quality.valid_fraction.min(), quality.valid_fraction.max())
download_outputs_from_colab(
    "eye_tracking_foundations/output", "01"
)
